In [1]:
import numpy as np
from typing import List
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
import copy
from qiskit.quantum_info import random_unitary
import matplotlib as mpl
import time
from line_profiler import LineProfiler
import pickle
import joblib

In [2]:
def cost_function1(n: int) -> int:
    return (4 - 3*2**(3-n))

In [3]:
def rz_gate(theta: float):
    return np.array([[np.exp(-1j * theta/2), 0], [0, np.exp(1j * theta/2)]])

In [4]:
# Creating the Tau gate sets
def odd_k(denom: int) -> List[int]:
    k_list = []
    highest = denom//2 - 1
    k_list = list(range(-highest, highest+1, 2))
    return k_list
    
def tau_set(n: int):
    tau = []
    denom = 2**(n-1)
    k_list = odd_k(denom)
    cost = cost_function1(n)
    for k in k_list:
        gate = rz_gate(k*np.pi/denom)
        tau.append({"gate": gate, "t_count": cost, "name": f"Rz({k}*π/{denom})", "magic_gate": True})
    return tau

In [5]:
# Creating the logical base gate set number 1
set1 = []

pauli_x = np.array([[0, 1], [1, 0]])
set1.append({"gate": pauli_x, "t_count": 0, "name": "X", "magic_gate": False})

pauli_y = np.array([[0, -1j], [1j, 0]])
set1.append({"gate": pauli_y, "t_count": 0, "name": "Y", "magic_gate": False})

pauli_z = np.array([[1, 0], [0, -1]])
set1.append({"gate": pauli_z, "t_count": 0, "name": "Z", "magic_gate": False})

hadamard = 1/np.sqrt(2) * np.array([[1, 1], [1, -1]])
set1.append({"gate": hadamard, "t_count": 0, "name": "H", "magic_gate": False})

s_gate = rz_gate(np.pi/2)
set1.append({"gate": s_gate, "t_count": 0, "name": "S", "magic_gate": False})

sdg_gate = rz_gate(-np.pi/2)
set1.append({"gate": sdg_gate, "t_count": 0, "name": "Sdg", "magic_gate": False})

set1 = set1 + tau_set(3)
print(set1)

[{'gate': array([[0, 1],
       [1, 0]]), 't_count': 0, 'name': 'X', 'magic_gate': False}, {'gate': array([[ 0.+0.j, -0.-1.j],
       [ 0.+1.j,  0.+0.j]]), 't_count': 0, 'name': 'Y', 'magic_gate': False}, {'gate': array([[ 1,  0],
       [ 0, -1]]), 't_count': 0, 'name': 'Z', 'magic_gate': False}, {'gate': array([[ 0.70710678,  0.70710678],
       [ 0.70710678, -0.70710678]]), 't_count': 0, 'name': 'H', 'magic_gate': False}, {'gate': array([[0.70710678-0.70710678j, 0.        +0.j        ],
       [0.        +0.j        , 0.70710678+0.70710678j]]), 't_count': 0, 'name': 'S', 'magic_gate': False}, {'gate': array([[0.70710678+0.70710678j, 0.        +0.j        ],
       [0.        +0.j        , 0.70710678-0.70710678j]]), 't_count': 0, 'name': 'Sdg', 'magic_gate': False}, {'gate': array([[0.92387953+0.38268343j, 0.        +0.j        ],
       [0.        +0.j        , 0.92387953-0.38268343j]]), 't_count': 1, 'name': 'Rz(-1*π/4)', 'magic_gate': True}, {'gate': array([[0.92387953-0.38268343j

In [6]:
def remove_global_phase(U):
    det = U[0,0]*U[1,1] - U[0,1]*U[1,0]
    return U / np.sqrt(det + 0j)

In [7]:
# Adding exotic magic state to set1
theta = np.arctan(np.sqrt((np.sqrt(5) - 1) / 2))
exotic_magic_state = np.array([[np.cos(theta/2), -np.sin(theta/2)], [np.sin(theta/2), np.cos(theta/2)]])
exotic_entry = {"gate": exotic_magic_state, "t_count": 0, "name": "EMS", "magic_gate": True}

exotic_set1 = set1.copy()
exotic_set1.append(exotic_entry)

In [8]:
gate_dict = {}
gate_dict['I'] = np.eye(2, dtype='complex')
gate_dict['X'] = np.array([[0, 1], [1, 0]])
gate_dict['Y'] = np.array([[0, -1j], [1j, 0]])
gate_dict['Z'] = np.array([[1, 0], [0, -1]])
gate_dict['H'] = 1/np.sqrt(2) * np.array([[1, 1], [1, -1]])
gate_dict['S'] = rz_gate(np.pi/2)
gate_dict['Sdg'] = rz_gate(-np.pi/2)
gate_dict['Rz(-1*π/4)'] = np.array([[0.92387953+0.38268343j, 0.        +0.j        ],
       [0.        +0.j        , 0.92387953-0.38268343j]])
gate_dict['Rz(1*π/4)'] = np.array([[0.92387953-0.38268343j, 0.        +0.j        ],
       [0.        +0.j        , 0.92387953+0.38268343j]])
gate_dict['EMS'] = np.array([[np.cos(theta/2), -np.sin(theta/2)], [np.sin(theta/2), np.cos(theta/2)]])

In [9]:
def key_from_U(U: np.ndarray):
    # U = aI + i(xX + yY + zZ)
    a, b, c, d = U[0,0], U[0,1], U[1,0], U[1,1]
    det = a*d - b*c
    s = 1.0 / np.sqrt(det + 0j)
    a *= s;  b *= s;  c *= s;  d *= s
    
    return (
        (0.5 * (a + d)).real,   # was: np.real( 0.5  * (a+d))
         0.5 * (b + c).imag,    # was: np.real(-0.5j * (b+c))
        (0.5 * (b - c)).real,   # was: np.real( 0.5  * (b-c))
         0.5 * (a - d).imag     # was: np.real(-0.5j * (a-d))
    )

In [ ]:
class SequenceDB:
    def __init__(self):
        self.vecs = {} # (a,x,y,z)
        self.seqs = [] # sequence of gates as a list of list of strings
        self.magic_count = [] # total magic count
        #self.t_count = [] # total t count
        self.unitaries = [] # 2x2 unitaries
        
    def add(self, v, seq, magic_count, #t_count, 
            matrix):
        self.vecs[v] = None
        self.seqs.append([str(g) for g in seq])
        self.magic_count.append(int(magic_count))
        #self.t_count.append(int(t_count))
        self.unitaries.append(np.array(matrix))
        
    def save(self, filepath):
        state = {
        "vecs": self.vecs,
        "seqs": self.seqs,
        "magic_count": np.asarray(self.magic_count, dtype=np.int16),
        #"t_count": np.asarray(self.t_count, dtype=np.int16),
        "unitaries": np.asarray(self.unitaries, dtype=np.complex128),
        }
        with open(filepath, 'wb') as f:
            pickle.dump(state, f, protocol=pickle.HIGHEST_PROTOCOL)

    @classmethod
    def load(cls, filepath) -> "SequenceDB":
        with open(filepath, 'rb') as f:
            return pickle.load(f)

In [ ]:
# Without t_count
def generate_sequences(base_gate_set, max_magic_count, previous=None):
    clifford_gates = [g for g in base_gate_set if not g['magic_gate']]
    magic_gates = [g for g in base_gate_set if g['magic_gate']]
    
    if previous is None:
        db = SequenceDB()
        db.add((0.0, 0.0, 0.0, 0.0), ['I'], 0.0, np.eye(2, dtype='complex'))
        for g in clifford_gates:
            db.add(key_from_U(g['gate']), [g['name']], 0.0, g['gate'])
        for g in magic_gates:
            db.add(key_from_U(g['gate']), [g['name']], 1.0, g['gate'])
    else:
        db = copy.deepcopy(previous)
        if max(db.magic_count) >= max_magic_count:
            print("Max Magic Count Reached")
            return db
    

    for i, seq in enumerate(db.seqs):
        # from seq, obtaining the 2x2 matrix
        current_unitary = db.unitaries[i]

        # last gate in sequence is a clifford? 
        clifford = False if (seq[-1] == 'Rz(-1*π/4)' or seq[-1] == 'Rz(1*π/4)' or seq[-1] == 'EMS') else True
        
        # looping over all possible gates, either only Clifford or only magic
        candidates = magic_gates if clifford else clifford_gates
        for gate_info in candidates:
            # one needs to be a magic gate and the other needs to be a clifford
            if gate_info['magic_gate'] != clifford:
                continue
            
            # if new unitary already exists skip, otherwise add to database
            temp_unitary = gate_info['gate'] @ current_unitary
            temp_vector = key_from_U(temp_unitary)
            if temp_vector in db.vecs:
                continue
            else:
                temp_magic_count = 1 if clifford == True else 0
                # check if the new unitary has exceeded the max number of magic states
                if db.magic_count[i] + temp_magic_count > max_magic_count:
                    continue
                else:
                    temp_seq = seq + [gate_info['name']]
                    db.add(temp_vector, temp_seq, db.magic_count[i] + temp_magic_count, temp_unitary)
    
    return db

In [ ]:
# With t_count
def generate_sequences(base_gate_set, max_magic_count, previous=None):
    clifford_gates = [g for g in base_gate_set if not g['magic_gate']]
    magic_gates = [g for g in base_gate_set if g['magic_gate']]
    
    if previous is None:
        db = SequenceDB()
        db.add((0.0, 0.0, 0.0, 0.0), ['I'], 0.0, 0.0, np.eye(2, dtype='complex'))
        for g in clifford_gates:
            db.add(key_from_U(g['gate']), [g['name']], 0.0, 0.0, g['gate'])
        for g in magic_gates:
            db.add(key_from_U(g['gate']), [g['name']], 1.0, 1.0, g['gate'])
    else:
        db = copy.deepcopy(previous)
        if max(db.magic_count) >= max_magic_count:
            print("Max Magic Count Reached")
            return db
    

    for i, seq in enumerate(db.seqs):
        # from seq, obtaining the 2x2 matrix
        current_unitary = db.unitaries[i]

        # last gate in sequence is a clifford? 
        clifford = False if (seq[-1] == 'Rz(-1*π/4)' or seq[-1] == 'Rz(1*π/4)' or seq[-1] == 'EMS') else True
        
        # looping over all possible gates, either only Clifford or only magic
        candidates = magic_gates if clifford else clifford_gates
        for gate_info in candidates:
            # one needs to be a magic gate and the other needs to be a clifford
            if gate_info['magic_gate'] != clifford:
                continue
            
            # if new unitary already exists skip, otherwise add to database
            temp_unitary = gate_info['gate'] @ current_unitary
            temp_vector = key_from_U(temp_unitary)
            if temp_vector in db.vecs:
                continue
            else:
                temp_magic_count = 1 if clifford == True else 0
                temp_t_count = 1 if (gate_info['name'] == 'Rz(-1*π/4)' or gate_info['name'] == 'Rz(1*π/4)') else 0
                # check if the new unitary has exceeded the max number of magic states
                if db.magic_count[i] + temp_magic_count > max_magic_count:
                    continue
                else:
                    temp_seq = seq + [gate_info['name']]
                    db.add(temp_vector, temp_seq, db.magic_count[i] + temp_magic_count, db.t_count[i] + temp_t_count, temp_unitary)
    
    return db

In [12]:
exotic_set1_max2 = generate_sequences(exotic_set1, 2)

In [14]:
exotic_set1_max5 = generate_sequences(exotic_set1, 5, exotic_set1_max2)

In [16]:
exotic_set1_max6 = generate_sequences(exotic_set1, 6, exotic_set1_max5)

In [ ]:
exotic_set1_max7 = generate_sequences(exotic_set1, 7, exotic_set1_max6)

In [ ]:
exotic_set1_max8 = generate_sequences(exotic_set1, 8, exotic_set1_max7)

In [ ]:
exotic_set1_max9 = generate_sequences(exotic_set1, 9, exotic_set1_max8)

In [ ]:
exotic_set1_max10 = generate_sequences(exotic_set1, 10, exotic_set1_max9)

In [ ]:
# Saving
exotic_set1_max6.save("exotic_set1_max6.pkl")

In [ ]:
# Loading
exotic_set1_max5 = SequenceDB.load("exotic_set1_max5.pkl")

In [ ]:
set1_max5 = generate_sequences(set1, 5)

In [ ]:
set1_max6 = generate_sequences(set1, 6, set1_max5)

In [ ]:
set1_max7 = generate_sequences(set1, 7, set1_max6)

In [ ]:
set1_max8 = generate_sequences(set1, 8, set1_max7)

Given an error tolerance epsilon, find an approximation within the error tolerance

In [110]:
def distance(U, V):
    # U and V both have the form (a, x, y, z)
    if np.array_equal(U, V):
        return 0
    else:
        return np.sqrt( (1 - abs(np.dot(U, V))) + 0j).real

In [ ]:
def db_to_nn(db):
    keys = list(db.vecs.keys())
    keys.pop(0)
    #V = np.asarray(keys)
    V = [np.array(k, dtype=float) for k in keys]
    V = np.array(V)
    
    nn = NearestNeighbors(algorithm="auto", metric="euclidean")
    nn.fit(V)
    return nn

In [147]:
def closest_neighbors(unitary, db, nn, epsilon):
    euclidean_dist, idxs = nn.kneighbors(np.array(key_from_U(unitary), dtype=float).reshape(1, -1), 50, return_distance=True)
    
    magic_count_list = []
    distance_list = []
    unitary_list = []
    seqs_list = []
    for i in idxs[0]:
        temp_distance = distance(np.array(key_from_U(db.unitaries[i]), dtype=float), np.array(key_from_U(unitary), dtype=float))
        if (temp_distance <= epsilon):
            magic_count_list.append(db.magic_count[i])
            distance_list.append(temp_distance)
            unitary_list.append(db.unitaries[i])
            seqs_list.append(db.seqs[i])
            
    if (len(magic_count_list) == 0):
        print(f"No closest neighbors given epsilon {epsilon}.")
        
    return magic_count_list, distance_list, unitary_list, seqs_list

In [ ]:
alpha1 = -0.274220
rz_alpha1_matrix = np.array([[np.exp(-1j*alpha1/2), 0], [0, np.exp(1j*alpha1/2)]])

In [73]:
enn8 = db_to_nn(exotic_set1_max8)

/var/folders/yh/ktq4n6y1305b9mr9r8_d4znw0000gn/T/ipykernel_27151/1995260033.py:5: ComplexWarning: Casting complex values to real discards the imaginary part
  V = [np.array(k, dtype=float) for k in keys]


In [162]:
magic_count_list, distance_list, unitary_list, seqs_list = closest_neighbors(rz_alpha1_matrix, exotic_set1_max8, enn8, 0.1)
print(len(magic_count_list))

1


In [144]:
print(magic_count_list)

[8.0]


In [145]:
print(unitary_list[0])

[[ 0.97757065+0.17342968j  0.10945499+0.04793094j]
 [-0.10945499+0.04793094j  0.97757065-0.17342968j]]


In [121]:
print(rz_alpha1_matrix)

[[0.99061514+0.13668081j 0.        +0.j        ]
 [0.        +0.j         0.99061514-0.13668081j]]


In [150]:
print(seqs_list)

[['S', 'EMS', 'X', 'Rz(1*π/4)', 'H', 'EMS', 'S', 'EMS', 'Sdg', 'Rz(-1*π/4)', 'Sdg', 'EMS', 'H', 'Rz(1*π/4)', 'H', 'Rz(1*π/4)']]


In [146]:
print(key_from_U(unitary_list[0]))
print(key_from_U(rz_alpha1_matrix))
print(distance(key_from_U(unitary_list[0]), key_from_U(rz_alpha1_matrix)))

[0.97757065 0.04793094 0.10945499 0.17342968]
[0.99061514 0.         0.         0.13668081]
0.08887746498260106
